In [4]:
# =========================
# CELL 1: Setup + Buffered POLY + Land Mask
# =========================

import ee
import geemap

# =========================
# Initialize Earth Engine
# =========================
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

# =========================
# USER SETTINGS
# =========================

BUFFER_DEG = 0.2


# Original Oahu bounds
min_lon = 124.137
max_lon = 127.764
min_lat = -9.739
max_lat = -7.619

# =========================
# Apply buffer
# =========================

min_lon_buffer = min_lon - BUFFER_DEG
max_lon_buffer = max_lon + BUFFER_DEG
min_lat_buffer = min_lat - BUFFER_DEG
max_lat_buffer = max_lat + BUFFER_DEG

POLY = [
    [min_lon_buffer, min_lat_buffer],
    [max_lon_buffer, min_lat_buffer],
    [max_lon_buffer, max_lat_buffer],
    [min_lon_buffer, max_lat_buffer],
    [min_lon_buffer, min_lat_buffer],
]

region_of_interest = ee.Geometry.Polygon(POLY)

# =========================
# Debug info
# =========================

print("========== BUFFERED POLY ==========")

print(
    f"Longitude: "
    f"{min_lon_buffer:.3f} → {max_lon_buffer:.3f}"
)

print(
    f"Latitude: "
    f"{min_lat_buffer:.3f} → {max_lat_buffer:.3f}"
)

# =========================
# Load ESA WorldCover
# =========================

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(region_of_interest)
)

# =========================
# Water mask
# class 80 = water
# =========================

water_mask = (
    worldcover
    .eq(80)
    .rename("water")
    .toByte()
)

# =========================
# Land mask
# everything except water
# =========================

land_mask = (
    worldcover
    .neq(80)
    .rename("land")
    .toByte()
)

# =========================
# Clean edges / tiny artifacts
# =========================

land_mask_clean = (
    land_mask
    .focal_max(
        radius=20,
        units="meters"
    )
    .focal_min(
        radius=20,
        units="meters"
    )
    .rename("land")
    .toByte()
)

print("\nLand mask generated successfully")

========== BUFFERED POLY ==========
Longitude: 123.937 → 127.964
Latitude: -9.939 → -7.419

Land mask generated successfully


In [5]:
# =========================
# CELL 2: Export Land Mask + Check Task Status
# =========================

import ee
import time

# =========================
# PARAMETERS
# =========================

EXPORT_DESCRIPTION = "land_mask_new_area_buffered"
EXPORT_FOLDER = "GEE_exports"
EXPORT_FILENAME = "land_mask_timor"

EXPORT_SCALE = 10          # ESA WorldCover resolution
EXPORT_CRS = "EPSG:4326"
MAX_PIXELS = 1e13

CHECK_STATUS_AFTER_START = True
WAIT_SECONDS = 5           # wait a few seconds before checking status

# =========================
# Initialize Earth Engine
# =========================

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

# =========================
# Export land mask to Google Drive
# =========================

task = ee.batch.Export.image.toDrive(
    image=land_mask_clean,
    description=EXPORT_DESCRIPTION,
    folder=EXPORT_FOLDER,
    fileNamePrefix=EXPORT_FILENAME,
    region=region_of_interest,
    scale=EXPORT_SCALE,
    crs=EXPORT_CRS,
    maxPixels=MAX_PIXELS
)

task.start()

TASK_ID = task.id

print("========== EXPORT STARTED ==========")
print("Description:", EXPORT_DESCRIPTION)
print("Folder:", EXPORT_FOLDER)
print("Filename:", EXPORT_FILENAME)
print("Task ID:", TASK_ID)

# =========================
# Optional: Check task status
# =========================

if CHECK_STATUS_AFTER_START:
    time.sleep(WAIT_SECONDS)

    tasks = ee.batch.Task.list()

    found = False

    for t in tasks:
        status = t.status()

        if status.get("id") == TASK_ID:
            found = True

            print("\n========== TASK STATUS ==========")
            print("Task ID:", status.get("id"))
            print("Description:", status.get("description"))
            print("State:", status.get("state"))
            print("Creation time:", status.get("creation_timestamp_ms"))
            print("Update time:", status.get("update_timestamp_ms"))

            if "error_message" in status:
                print("Error:", status.get("error_message"))

            break

    if not found:
        print("\nTask ID not found in recent task list.")
        print("It may still be initializing, or Earth Engine task list has not refreshed yet.")

========== EXPORT STARTED ==========
Description: land_mask_new_area_buffered
Folder: GEE_exports
Filename: land_mask_timor
Task ID: F2URBJ7Z2BZQVZB2GXWH5KS5

========== TASK STATUS ==========
Task ID: F2URBJ7Z2BZQVZB2GXWH5KS5
Description: land_mask_new_area_buffered
State: RUNNING
Creation time: 1779898744697
Update time: 1779898749568
